# Gas Turbine

This dataset contains techno-economic data on a gas turbines (conventional, hydrogen-ready, conventional with carbon capture).

In [1]:
# Dependencies.
from IPython.display import HTML, Markdown

import numpy as np
import pandas as pd
pd.options.plotting.backend = "plotly"
import plotly.express as px
from plotly.subplots import make_subplots

from posted import TEDF


# Set variable of TEDF.
var = "Tech|Gas Turbine"

# Loading the TEDF.
tedf = TEDF.load(var)

# Define units to use.
units = {
    "Output Capacity|Hydrogen": "MW_H2_LHV",
    "CAPEX": "EUR_2024",
    "OPEX Fixed": "EUR_2024/yr",
    "OPEX Variable": "EUR_2024",
    "Output|Electricity": "MWh",
    "Output Capacity|Electricity": "MW",
}

## Fields

The techno-economic data is distinguished across the following fields.

### Subtechnologies (`subtech`)

In [2]:
Markdown(
    "\n".join(f"* **{code}**: {desc}" for code, desc in tedf.fields["subtech"].codes.items())
)

* **CCGT conventional**: Conventional CCGT w/o carbon capture and non-H2-ready.
* **CCGT H2-ready**: CCGT that is H2-ready.
* **CCGT CC**: CCGT with carbon capture.

### Hydrogen operation (`mode_h2`)

In [3]:
Markdown(
    "\n".join(f"* **{code}**: {desc}" for code, desc in tedf.fields["mode_h2"].codes.items())
)

* **NG**: 0% hydrogen, 100% natural gas.
* **H2**: 100% hydrogen, 0% natural gas.

## Aggregated parameters

All data in this dataset can be aggregated via the NOSLAG workflow, which yields the following parameters:

In [4]:
aggregated = tedf.aggregate(units=units, append_references=True)

display(
    aggregated
    .pivot(
        index=aggregated.columns[:-3],
        columns=["variable", "unit"],
        values="value",
    )
    # Drop hydrogen operation mode for subtechnologies where it doesn't make sense.
    .query("mode_h2!='H2' or subtech=='CCGT H2-ready'")
    .map(lambda x: float(f"{x:.3g}") if not pd.isnull(x) else x)
    .fillna("")
    .sort_index(key=lambda idx: pd.Series(idx).apply(list(tedf.fields[idx.name].codes).index))
)

variable                  Output Capacity|Electricity Output|Electricity  \
unit                                               MW                MWh   
subtech           mode_h2                                                  
CCGT conventional NG                              1.0                1.0   
CCGT H2-ready     NG                              1.0                1.0   
                  H2                              1.0                1.0   
CCGT CC           NG                              1.0                1.0   

variable                       CAPEX Capture Rate Input|Natural Gas  \
unit                        EUR_2024      percent        kWh_NG_LHV   
subtech           mode_h2                                             
CCGT conventional NG       1160000.0                         1670.0   
CCGT H2-ready     NG       1340000.0                         1670.0   
                  H2       1340000.0                            0.0   
CCGT CC           NG       2260000.0         90.0            2170.0   

variable                   OPEX Fixed OPEX Variable Input|Hydrogen  
unit                      EUR_2024/yr      EUR_2024     kWh_H2_LHV  
subtech           mode_h2                                           
CCGT conventional NG          21600.0                               
CCGT H2-ready     NG          21600.0                          0.0  
                  H2          21600.0                       1670.0  
CCGT CC           NG          52800.0          1.31

## Raw data

In [5]:
Markdown(f"""
The table below contains the raw data contained in this dataset. The raw data has not be normalised or harmonised 
in any way and should closely resemble the data as it is reported by the respective sources. You can also find 
this data in the GitHub repo in this file:
{link_public_github(var)}
""")


The table below contains the raw data contained in this dataset. The raw data has not be normalised or harmonised 
in any way and should closely resemble the data as it is reported by the respective sources. You can also find 
this data in the GitHub repo in this file:
<a href="https://github.com/PhilippVerpoort/posted/blob/main/posted/database/tedfs/Tech/Gas Turbine.csv">posted/database/tedfs/Tech/Gas Turbine.csv</a>


In [6]:
tedf.edit_data()